# 🧠 Taller SQL - Gimnasia Rítmica con PostgreSQL

Este taller está diseñado para ejecutarse en Google Colaboratory usando una base de datos PostgreSQL. Aprenderás a crear tablas, insertar datos y realizar consultas SQL avanzadas sobre una base de datos que modela competencias de gimnasia rítmica.

## Objetivos
- Crear una base de datos PostgreSQL en Colab.
- Crear tablas con claves primarias y foráneas.
- Insertar datos simulados.
- Ejecutar consultas SQL avanzadas.
- Resolver ejercicios prácticos.


In [1]:
# Instalación y configuración de PostgreSQL
!apt update
!apt install postgresql postgresql-contrib -y
!service postgresql start
!sudo -u postgres psql -c "CREATE USER root WITH SUPERUSER"


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,006 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Get:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,267 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/pp

In [2]:
# Cargar extensión SQL y conectar con PostgreSQL
%load_ext sql
%config SqlMagic.feedback=False
%config SqlMagic.autopandas=True
%sql postgresql+psycopg2://@/postgres


## 🏗️ Crear tablas

Creamos las tablas necesarias para modelar la base de datos de gimnasia rítmica. Incluye campos tipo `DATE` y relaciones entre entidades.


In [3]:
%%sql

DROP TABLE IF EXISTS Evaluaciones, Presentaciones, Gimnastas, Conjuntos, Instrumentos, Campeonatos, Tipos, Equipos CASCADE;

CREATE TABLE Equipos (
    id_equipo SERIAL PRIMARY KEY,
    nombre_equipo TEXT
);

CREATE TABLE Conjuntos (
    id_conj SERIAL PRIMARY KEY,
    nombre_conj TEXT,
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Gimnastas (
    id_gim SERIAL PRIMARY KEY,
    nombre TEXT,
    fecha_nac DATE,
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Instrumentos (
    id_inst SERIAL PRIMARY KEY,
    nombre_inst TEXT
);

CREATE TABLE Campeonatos (
    id_camp SERIAL PRIMARY KEY,
    nombre_camp TEXT,
    fecha_camp DATE
);

CREATE TABLE Tipos (
    id_tipo SERIAL PRIMARY KEY,
    nombre_tipo TEXT
);

CREATE TABLE Presentaciones (
    id_pres SERIAL PRIMARY KEY,
    id_gim INTEGER REFERENCES Gimnastas(id_gim),
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_inst INTEGER REFERENCES Instrumentos(id_inst),
    id_camp INTEGER REFERENCES Campeonatos(id_camp),
    id_tipo INTEGER REFERENCES Tipos(id_tipo)
);

CREATE TABLE Evaluaciones (
    id_eval SERIAL PRIMARY KEY,
    id_pres INTEGER REFERENCES Presentaciones(id_pres),
    pje_eje REAL,
    pje_dif REAL,
    pje_art REAL
);


 * postgresql+psycopg2://@/postgres


""


In [4]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


## 📥 Insertar datos simulados

Insertamos datos ficticios para poblar la base de datos: 5 equipos, 10 conjuntos, 20 gimnastas, 15 campeonatos, 50 presentaciones individuales, 50 presentaciones de conjunto y 100 evaluaciones.


In [5]:
%%sql

-- Insertar equipos
INSERT INTO Equipos (nombre_equipo) VALUES
('Estrellas'), ('Auroras'), ('Fénix'), ('Orion'), ('Galaxia');

-- Insertar conjuntos
INSERT INTO Conjuntos (nombre_conj, id_equipo) VALUES
('Estrellas A', 1), ('Estrellas B', 1), ('Auroras A', 2), ('Auroras B', 2),
('Fénix A', 3), ('Fénix B', 3), ('Orion A', 4), ('Orion B', 4),
('Galaxia A', 5), ('Galaxia B', 5);

-- Insertar gimnastas
INSERT INTO Gimnastas (nombre, fecha_nac, id_conj, id_equipo) VALUES
('Ana Torres', '2005-03-12', 1, 1), ('Lucía Pérez', '2006-07-25', 1, 1),
('María Gómez', '2004-11-05', 2, 1), ('Sofía Díaz', '2007-01-20', 2, 1),
('Valentina Ruiz', '2005-06-30', 3, 2), ('Camila Soto', '2006-09-15', 3, 2),
('Isabella León', '2004-12-01', 4, 2), ('Martina Ríos', '2007-04-10', 4, 2),
('Renata Silva', '2005-08-22', 5, 3), ('Emilia Vargas', '2006-10-05', 5, 3),
('Josefa Herrera', '2004-07-18', 6, 3), ('Antonia Castro', '2007-02-28', 6, 3),
('Florencia Peña', '2005-05-14', 7, 4), ('Amanda Fuentes', '2006-11-11', 7, 4),
('Julieta Navarro', '2004-09-09', 8, 4), ('Agustina Bravo', '2007-03-03', 8, 4),
('Daniela Pino', '2005-12-25', 9, 5), ('Bianca Morales', '2006-06-06', 9, 5),
('Carla Espinoza', '2004-10-10', 10, 5), ('Fernanda Reyes', '2007-07-07', 10, 5),
('Colomba Gómez', '2004-11-05', 2, 1), ('Montserrat Díaz', '2007-01-20', 2, 1);

-- Insertar instrumentos
INSERT INTO Instrumentos (nombre_inst) VALUES
('Cinta'), ('Aro'), ('Balón'), ('Manos libres'), ('Cuerda');

-- Insertar campeonatos
INSERT INTO Campeonatos (nombre_camp, fecha_camp) VALUES
('Campeonato Nacional', '2023-06-15'), ('Copa Primavera', '2023-09-10'),
('Copa Invierno', '2023-12-05'), ('Torneo Sur', '2023-08-20'),
('Copa Norte', '2023-07-01'), ('Copa Andes', '2023-10-10'),
('Copa Pacífico', '2023-11-11'), ('Copa Estrellas', '2024-05-05'),
('Copa Aurora', '2024-04-04'), ('Copa Fénix', '2025-03-03'),
('Copa Orion', '2025-02-02'), ('Copa Galaxia', '2025-01-01'),
('Copa Final', '2024-12-31'), ('Copa Apertura', '2025-01-15'),
('Copa Clausura', '2024-12-01');

-- Insertar tipos
INSERT INTO Tipos (nombre_tipo) VALUES ('Individual'), ('Conjunto');




 * postgresql+psycopg2://@/postgres


""


In [6]:
%%sql
-- Insertar presentaciones
DO $$
DECLARE
    i INTEGER := 1;
    gim_id INTEGER;
    conj_id INTEGER;
    inst_id INTEGER;
    tipo_id INTEGER;
    camp_id INTEGER;
BEGIN
    WHILE i <= 100 LOOP
        -- Selección aleatoria de IDs válidos
        SELECT id_gim INTO gim_id FROM Gimnastas ORDER BY RANDOM() LIMIT 1;
        SELECT id_conj INTO conj_id FROM Conjuntos ORDER BY RANDOM() LIMIT 1;
        SELECT id_inst INTO inst_id FROM Instrumentos ORDER BY RANDOM() LIMIT 1;
        SELECT id_tipo INTO tipo_id FROM Tipos ORDER BY RANDOM() LIMIT 1;
        SELECT id_camp INTO camp_id FROM Campeonatos ORDER BY RANDOM() LIMIT 1;

        -- Inserción en la tabla Presentaciones
        INSERT INTO Presentaciones (id_gim, id_conj, id_inst, id_camp, id_tipo)
        VALUES (gim_id, conj_id, inst_id, camp_id, tipo_id);

        i := i + 1;
    END LOOP;
END $$;


 * postgresql+psycopg2://@/postgres


""


In [7]:
%%sql
-- Insertar evaluaciones
INSERT INTO Evaluaciones (id_pres, pje_eje, pje_dif, pje_art)
SELECT p.id_pres,
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2)
FROM Presentaciones p
LIMIT 100;

 * postgresql+psycopg2://@/postgres


""


## 🔍 Ejercicios de Consultas SQL



1.- Gimnastas con más de 2 presentaciones individuales

In [9]:
%%sql
select g.nombre, count(*) as num_presentaciones
from presentaciones p inner join gimnastas g on p.id_gim = g.id_gim
where p.id_tipo = 1
group by p.id_gim,g.nombre
having count(*) > 2;

 * postgresql+psycopg2://@/postgres


,nombre,num_presentaciones
0,Emilia Vargas,5
1,Montserrat Díaz,5
2,Colomba Gómez,4
3,Camila Soto,4
4,Florencia Peña,6
5,María Gómez,4
6,Carla Espinoza,7


2.- Conjuntos con más de 3 presentaciones

In [10]:
%%sql
select c.nombre_conj, count(*) as num_presentaciones
from conjuntos c inner join presentaciones p on c.id_conj = p.id_conj
group by c.nombre_conj
having count(*) > 3;

 * postgresql+psycopg2://@/postgres


,nombre_conj,num_presentaciones
0,Galaxia B,7
1,Orion B,5
2,Auroras A,10
3,Auroras B,8
4,Fénix A,15
5,Orion A,10
6,Galaxia A,14
7,Fénix B,17
8,Estrellas B,11


3.- Campeonatos en el segundo semestre del año 2023.

In [11]:
%%sql
select * from campeonatos
where fecha_camp between '2023-07-01' and '2023-12-31';

 * postgresql+psycopg2://@/postgres


,id_camp,nombre_camp,fecha_camp
0,2,Copa Primavera,2023-09-10
1,3,Copa Invierno,2023-12-05
2,4,Torneo Sur,2023-08-20
3,5,Copa Norte,2023-07-01
4,6,Copa Andes,2023-10-10
5,7,Copa Pacífico,2023-11-11


4.- Gimnastas ordenadas por fecha de nacimiento, desde la mayor a la menor.

In [12]:
%%sql
select * from gimnastas
order by fecha_nac desc;

 * postgresql+psycopg2://@/postgres


,id_gim,nombre,fecha_nac,id_conj,id_equipo
0,20,Fernanda Reyes,2007-07-07,10,5
1,8,Martina Ríos,2007-04-10,4,2
2,16,Agustina Bravo,2007-03-03,8,4
3,12,Antonia Castro,2007-02-28,6,3
4,22,Montserrat Díaz,2007-01-20,2,1
5,4,Sofía Díaz,2007-01-20,2,1
6,14,Amanda Fuentes,2006-11-11,7,4
7,10,Emilia Vargas,2006-10-05,5,3
8,6,Camila Soto,2006-09-15,3,2
9,2,Lucía Pérez,2006-07-25,1,1


5.- Promedio del puntaje total (suma del puntaje de ejecución, artístico y dificultad) por año y tipo de presentación.

In [17]:
%%sql
select avg(e.pje_eje + e.pje_art + e.pje_dif) as puntaje_total, date_part ('year', c.fecha_camp) as año, p.id_tipo as tipo_presentacion
from evaluaciones e inner join presentaciones p on e.id_pres = p.id_pres
inner join campeonatos c on p.id_camp = c.id_camp
group by date_part('year', c.fecha_camp), p.id_tipo;

 * postgresql+psycopg2://@/postgres


,puntaje_total,año,tipo_presentacion
0,21.800770,2025.0,1
1,22.629375,2024.0,2
2,22.200417,2023.0,1
3,22.156666,2024.0,1
4,22.266522,2023.0,2
5,22.551111,2025.0,2


6.- Máximo puntaje artístico por año y tipo de presentación

In [19]:
%%sql
select max(e.pje_art) as puntaje_art, date_part ('year', c.fecha_camp) as año, p.id_tipo as tipo_presentacion
from evaluaciones e inner join presentaciones p on e.id_pres = p.id_pres
inner join campeonatos c on p.id_camp = c.id_camp
group by date_part('year', c.fecha_camp), p.id_tipo;

 * postgresql+psycopg2://@/postgres


,puntaje_art,año,tipo_presentacion
0,9.48,2025.0,1
1,9.52,2024.0,2
2,9.40,2023.0,1
3,9.58,2024.0,1
4,9.99,2023.0,2
5,9.08,2025.0,2


7.- Gimnastas que participaron en el campeonato más reciente

In [20]:
%%sql
select distinct g.nombre
from gimnastas g inner join presentaciones p on g.id_gim = p.id_gim
inner join campeonatos c on p.id_camp = c.id_camp
where c.fecha_camp = (select max(fecha_camp) from campeonatos);

 * postgresql+psycopg2://@/postgres


,nombre
0,Agustina Bravo
1,Antonia Castro
2,Bianca Morales
3,Fernanda Reyes
4,Florencia Peña
5,Isabella León


8.- Equipos que tienen más de 5 gimnastas

In [21]:
%%sql
select e.nombre_equipo, count(*) as num_gimnastas
from equipos e inner join gimnastas g on e.id_equipo = g.id_equipo
group by e.nombre_equipo
having count(*) > 5;

 * postgresql+psycopg2://@/postgres


,nombre_equipo,num_gimnastas
0,Estrellas,6


9.- Presentaciones individuales con el puntaje total más alto por año e instrumento.

In [22]:
%%sql
select max(e.pje_eje + e.pje_art + e.pje_dif) as puntaje_total, date_part ('year', c.fecha_camp) as año, p.id_inst as instrumento
from evaluaciones e inner join presentaciones p on e.id_pres = p.id_pres
inner join campeonatos c on p.id_camp = c.id_camp
where p.id_tipo = 1
group by date_part('year', c.fecha_camp), p.id_inst;

 * postgresql+psycopg2://@/postgres


,puntaje_total,año,instrumento
0,25.540000,2025.0,1
1,24.700000,2025.0,3
2,25.290000,2024.0,2
3,25.490000,2024.0,4
4,19.160000,2025.0,5
5,24.320000,2023.0,1
6,26.500000,2023.0,3
7,26.160000,2024.0,1
8,24.970000,2023.0,2
9,25.079998,2024.0,5


10.- Presentaciones de conjunto con el puntaje total más alto por año e instrumento.

In [23]:
%%sql
select max(e.pje_eje + e.pje_art + e.pje_dif) as puntaje_total, date_part ('year', c.fecha_camp) as año, p.id_inst as instrumento
from evaluaciones e inner join
presentaciones p on e.id_pres = p.id_pres
inner join campeonatos c on p.id_camp = c.id_camp
where p.id_tipo = 2
group by date_part('year', c.fecha_camp), p.id_inst;

 * postgresql+psycopg2://@/postgres


,puntaje_total,año,instrumento
0,23.980000,2025.0,1
1,24.910000,2024.0,2
2,24.570000,2024.0,4
3,27.860000,2025.0,5
4,23.099998,2023.0,1
5,23.040000,2023.0,3
6,23.780000,2024.0,1
7,24.610000,2023.0,2
8,27.450000,2024.0,5
9,25.100000,2023.0,5


11.- Listar los nombres de los equipos y el promedio del puntaje artístico de todas las presentaciones (tanto individuales como por conjuntos) por año

In [34]:
%%sql
select ptos_equipos.nombre_equipo, avg(ptos_equipos.puntaje_art) as puntaje_art, año
from
(select eq.nombre_equipo, avg(e.pje_art) as puntaje_art, date_part ('year', c.fecha_camp) as año
from equipos eq inner join gimnastas g on eq.id_equipo = g.id_equipo
inner join presentaciones p on g.id_gim = p.id_gim
inner join campeonatos c on p.id_camp = c.id_camp
inner join evaluaciones e on p.id_pres = e.id_pres
group by eq.nombre_equipo  , date_part('year', c.fecha_camp)
union
select eq.nombre_equipo, avg(e.pje_art) as puntaje_art, date_part ('year', c.fecha_camp) as año
from equipos eq inner join conjuntos c2 on eq.id_equipo = c2.id_equipo
inner join presentaciones p on c2.id_conj = p.id_conj
inner join campeonatos c on p.id_camp = c.id_camp
inner join evaluaciones e on p.id_pres = e.id_pres
group by eq.nombre_equipo  , date_part('year', c.fecha_camp)) as ptos_equipos
group by nombre_equipo, año
order by nombre_equipo, año, puntaje_art;


 * postgresql+psycopg2://@/postgres


,nombre_equipo,puntaje_art,año
0,Auroras,7.670694,2023.0
1,Auroras,7.734167,2024.0
2,Auroras,6.333750,2025.0
3,Estrellas,7.228284,2023.0
4,Estrellas,7.390143,2024.0
5,Estrellas,7.895000,2025.0
6,Fénix,6.722976,2023.0
7,Fénix,7.190250,2024.0
8,Fénix,6.936875,2025.0
9,Galaxia,7.653333,2023.0
